# 🎓 World University Rankings Analysis
**CWUR Dataset — 2,200 universities · 59 countries · 2012–2015**

---

In [ ]:
import pandas as pd, numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings('ignore')
print("Libraries loaded")

## 1. Load Dataset

In [ ]:
df = pd.read_csv("cwurData.csv")
df.columns = [c.strip().lower() for c in df.columns]
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Years: {sorted(df['year'].unique())}")
print(f"Countries: {df['country'].nunique()}")
df.head(10)

## 2. Focus on 2015 Data

In [ ]:
df15 = df[df["year"] == 2015].copy()
print(f"2015 universities: {len(df15)}")
print(f"Countries represented: {df15['country'].nunique()}")
print("\nScore statistics:")
print(df15["score"].describe().round(2))

## 3. World Map — Universities per Country

In [ ]:
country_stats = df15.groupby("country").agg(
    Universities=("institution","count"),
    Avg_Score=("score","mean"),
    Best_Rank=("world_rank","min"),
).reset_index()
country_stats["Avg_Score"] = country_stats["Avg_Score"].round(2)

fig = px.choropleth(country_stats, locations="country", locationmode="country names",
    color="Universities", hover_name="country",
    hover_data=["Avg_Score","Best_Rank"],
    color_continuous_scale="Blues",
    title="Number of Universities in Top 1000 by Country (2015)")
fig.update_layout(height=500)
fig.show()

## 4. Top Countries

In [ ]:
top_countries = df15.groupby("country").agg(
    Count=("institution","count"),
    Avg_Score=("score","mean"),
).nlargest(20,"Count").reset_index()

fig = px.bar(top_countries, x="Count", y="country", orientation="h",
             color="Avg_Score", color_continuous_scale="Blues",
             title="Top 20 Countries — University Count (colored by Avg Score)")
fig.update_layout(height=550, yaxis=dict(autorange="reversed"))
fig.show()

## 5. Score Distribution

In [ ]:
fig = make_subplots(rows=1, cols=2)
fig.add_trace(go.Histogram(x=df15["score"], nbinsx=40,
    name="All", marker_color="steelblue"), row=1, col=1)

# Top 5 countries score distribution
top5 = df15["country"].value_counts().head(5).index
df_top5 = df15[df15["country"].isin(top5)]
for country in top5:
    sub = df_top5[df_top5["country"]==country]
    fig.add_trace(go.Box(y=sub["score"], name=country), row=1, col=2)

fig.update_layout(height=450,
    title="Score Distribution — All (left) & Top 5 Countries (right)",
    showlegend=True)
fig.show()

## 6. Metric Correlations

In [ ]:
score_cols = [c for c in ["quality_of_education","alumni_employment",
    "quality_of_faculty","publications","influence","citations",
    "broad_impact","patents","score"] if c in df15.columns]
corr = df15[score_cols].corr()
fig = px.imshow(corr, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                text_auto=".2f", title="Ranking Metric Correlations", aspect="auto")
fig.update_layout(height=520)
fig.show()

## 7. Scatter — Education Quality vs Score

In [ ]:
if "quality_of_education" in df15.columns:
    fig = px.scatter(df15, x="quality_of_education", y="score",
        color="country", hover_name="institution", trendline="ols",
        title="Education Quality Rank vs Overall Score",
        labels={"quality_of_education":"Education Quality Rank","score":"Overall Score"})
    fig.update_layout(height=500, showlegend=False)
    fig.show()

## 8. Radar Chart — Top 5 Universities

In [ ]:
top5u = df15.nsmallest(5, "world_rank")
radar_cols = [c for c in ["quality_of_education","alumni_employment",
    "quality_of_faculty","publications","influence","citations"] if c in df15.columns]

fig = go.Figure()
for _, row in top5u.iterrows():
    vals = [1 - row[c]/df15[c].max() for c in radar_cols]
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]], theta=radar_cols + [radar_cols[0]],
        fill="toself", name=row["institution"][:30]))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0,1])),
    title="Top 5 Universities — Metric Radar (1 = best)", height=500)
fig.show()

## 9. Year-over-Year Trend (Top Countries)

In [ ]:
top6_countries = df["country"].value_counts().head(6).index
trend = df[df["country"].isin(top6_countries)].groupby(["year","country"])["score"].mean().reset_index()
fig = px.line(trend, x="year", y="score", color="country", markers=True,
              title="Average Score Trend by Country (2012–2015)")
fig.update_layout(hovermode="x unified", height=420)
fig.show()

## 10. Key Insights

In [ ]:
print("=== UNIVERSITIES KEY INSIGHTS ===")
top1 = df15.nsmallest(1,"world_rank")
print(f"#1 University (2015): {top1['institution'].values[0]} ({top1['country'].values[0]})")
print(f"#1 Score            : {top1['score'].values[0]}")
top_c = country_stats.nlargest(1,"Universities")
print(f"Most universities   : {top_c['country'].values[0]} ({int(top_c['Universities'].values[0])})")
print(f"Universities >= 70  : {(df15['score']>=70).sum()}")
print(f"Countries tracked   : {df15['country'].nunique()}")
print("\nTop 5 by avg score:")
print(df15.groupby("country")["score"].mean().nlargest(5).round(2))